# Signal Fidelity Check — CSV signal vs cBot order execution

**Mục đích**: kiểm định "signal Python xuất ra" có được cBot (Combo / MA Cross)
nhận đúng và đặt lệnh hợp lệ hay không — tách biệt khỏi câu hỏi hiệu suất
($ lời/lỗ). Không tự tin vào bộ đếm `OnStop` của cBot — đối chiếu ĐỘC LẬP từ
2 nguồn thô: file CSV signal gốc và `log.txt` của lượt backtest đã archive.

**Bối cảnh hệ thống** (xem `AGENT.md` để biết đầy đủ):
- **DP6** (10.11.12.6): TradingView (kênh Capital.com) → SQL Server
  `SEN05_AutoTrading.DWH.Fact_OHLCV` (BarTime UTC-naive).
- **OG8** (10.11.12.8): `core_python` đọc `Fact_OHLCV` qua
  `db_connector.load_range()`, tính indicator + signal (`combo.py`/
  `ma_cross.py`), tính entry/SL/TP (`levels.py`), xuất CSV tối giản qua
  `export_cli.py` (`bartime,atr,signal` hoặc `bartime,atr,entry,signal`).
- **BO20** (máy này): cTrader đọc CSV, backtest/optimize
  (`Combo.cs`/`MA Cross.cs`) — đối tượng kiểm định của notebook này.

**Cách dùng cho lần sau**: sửa `RUNS` ở Cell "Chạy trên dữ liệu thật" trỏ
tới CSV signal + folder `ArchivedRuns\<Tên>` muốn kiểm định, chạy lại toàn
bộ notebook (Kernel → Restart & Run All).

## 1. Hàm lõi — đọc CSV, đọc log, đối chiếu

In [1]:
import re
import json
from pathlib import Path

import pandas as pd

# --- Regex khớp đúng 3 dạng dòng log mà Combo.cs/MA Cross.cs in ra ---
# (xem PlacePendingOrder/PlaceMarketOrder trong Combo.cs, MA Cross.cs)
PLACED_RE = re.compile(
    r"bartime=(?P<bartime>\S+ \S+), alignment=(?P<alignment>\w+), executed=(?P<executed>\S+ \S+), "
    r"(?:pending|market) (?P<direction>Buy|Sell) placed(?: at (?P<entry>[\d.]+))?;"
)
REJECTED_RE = re.compile(
    r"bartime=(?P<bartime>\S+ \S+), alignment=(?P<alignment>\w+), "
    r"(?:pending \S+ at [\d.]+ |market \S+ )was rejected: (?P<error>\S+)\."
)
FALLBACK_EXPIRED_RE = re.compile(
    r"bartime=(?P<bartime>\S+ \S+) has no exact FTMO bar and expired at (?P<expired_at>\S+ \S+) "
    r"before a tradable tick arrived\."
)
# Lỗi dưới sàn volume KHÔNG gắn bartime/alignment trong log (giới hạn đã biết
# của Combo.cs/MA Cross.cs hiện tại) — chỉ đếm được TỔNG số, không truy được
# về đúng tín hiệu nào. Xem CalculateVolume trong 2 file .cs.
VOLUME_FLOOR_RE = re.compile(r"risk calculates to [\d.]+ units, below broker minimum")


def load_signal_csv(path: str) -> pd.DataFrame:
    """Đọc nguyên CSV signal (Combo: bartime,atr,entry,signal; MA Cross: bartime,atr,signal)."""
    df = pd.read_csv(path)
    df["bartime"] = pd.to_datetime(df["bartime"])
    return df


def parse_log(log_path: str) -> tuple[pd.DataFrame, int]:
    """Đọc log.txt, trích ra 1 dòng/sự kiện cho mỗi bartime có xuất hiện trong log."""
    text = Path(log_path).read_text(encoding="utf-8", errors="replace")
    records: list[dict] = []
    for m in PLACED_RE.finditer(text):
        records.append({
            "bartime": m.group("bartime"), "alignment": m.group("alignment"),
            "outcome": "placed", "log_direction": m.group("direction"),
            "log_entry": m.group("entry"), "error": None,
        })
    for m in REJECTED_RE.finditer(text):
        records.append({
            "bartime": m.group("bartime"), "alignment": m.group("alignment"),
            "outcome": "rejected", "log_direction": None,
            "log_entry": None, "error": m.group("error"),
        })
    for m in FALLBACK_EXPIRED_RE.finditer(text):
        records.append({
            "bartime": m.group("bartime"), "alignment": "FallbackAligned",
            "outcome": "fallback_expired_waiting", "log_direction": None,
            "log_entry": None, "error": None,
        })
    df = pd.DataFrame.from_records(records)
    if not df.empty:
        df["bartime"] = pd.to_datetime(df["bartime"])
    n_untraceable = len(VOLUME_FLOOR_RE.findall(text))
    return df, n_untraceable


def get_run_window(run_dir: str) -> tuple[pd.Timestamp | None, pd.Timestamp | None]:
    """Suy ra khung thời gian THẬT của lượt backtest từ chính events.json (không tin vào input người dùng gõ)."""
    events = json.loads((Path(run_dir) / "events.json").read_text(encoding="utf-8"))
    times = sorted(e["time"] for e in events if e.get("time") is not None)
    if not times:
        return None, None
    start = pd.to_datetime(times[0], unit="ms", utc=True).tz_localize(None)
    end = pd.to_datetime(times[-1], unit="ms", utc=True).tz_localize(None)
    return start, end


In [2]:
def build_fidelity_report(signal_csv_path: str, archived_run_dir: str, strategy: str):
    """Đối chiếu 1 file CSV signal với 1 lượt backtest đã archive.

    Trả về (summary: dict, merged: DataFrame — 1 dòng/tín hiệu CSV, có cột
    'status' phân loại rõ số phận từng tín hiệu).
    """
    run_dir = Path(archived_run_dir)
    signals = load_signal_csv(signal_csv_path)
    log_events, n_untraceable = parse_log(str(run_dir / "log.txt"))
    start, end = get_run_window(run_dir)

    # Đọc parameters.cbotset để biết fallback có đang BẬT không — nếu TẮT,
    # 1 tín hiệu "trong khung test nhưng không có trong log" là ĐÚNG THIẾT KẾ
    # (cBot không hề cố xử lý), không phải cảnh báo thật. Xem
    # ProcessFallbackSignals trong Combo.cs/MA Cross.cs — return ngay nếu
    # !_fallbackReady, không log gì cho các tín hiệu bị bỏ qua kiểu này.
    params_path = run_dir / "parameters.cbotset"
    fallback_enabled = True
    if params_path.exists():
        params = json.loads(params_path.read_text(encoding="utf-8"))
        raw = params.get("Parameters", {}).get("EnableMissingBarFallback")
        if raw is not None:
            fallback_enabled = str(raw).strip().lower() in ("true", "1")

    merged = signals.merge(log_events, on="bartime", how="left")
    merged["status"] = merged["outcome"].fillna("not_found_in_log")

    if start is not None:
        in_window = (merged["bartime"] >= start) & (merged["bartime"] <= end)
        still_missing = merged["status"] == "not_found_in_log"
        # Ngoài khung test — đúng thiết kế (loaded=... nhưng chưa tới lượt).
        merged.loc[~in_window & still_missing, "status"] = "before_test_window"
        if fallback_enabled:
            # Fallback ĐANG BẬT mà vẫn thiếu trong khung test — CẢNH BÁO THẬT.
            merged.loc[in_window & still_missing, "status"] = "⚠ IN_WINDOW_BUT_MISSING"
        else:
            # Fallback TẮT — thiếu ở đây là chủ đích (không cố xử lý), không phải lỗi.
            merged.loc[in_window & still_missing, "status"] = "in_window_no_exact_bar_fallback_off"

    def _check_direction(row):
        if row["status"] != "placed":
            return None
        expect = "Buy" if row["signal"] == 1 else "Sell"
        return expect == row["log_direction"]

    merged["direction_ok"] = merged.apply(_check_direction, axis=1)

    if strategy == "combo" and "entry" in merged.columns:
        def _check_entry(row):
            if row["status"] != "placed" or pd.isna(row["log_entry"]):
                return None
            return abs(float(row["entry"]) - float(row["log_entry"])) < 0.01
        merged["entry_ok"] = merged.apply(_check_entry, axis=1)

    summary = {
        "strategy": strategy,
        "signal_csv": Path(signal_csv_path).name,
        "archived_run": Path(archived_run_dir).name,
        "run_window": f"{start} -> {end}" if start is not None else "unknown",
        "total_signals_in_csv": len(signals),
        **merged["status"].value_counts().to_dict(),
        "direction_mismatches": int((merged["direction_ok"] == False).sum()),  # noqa: E712
        "untraceable_volume_floor_skips": n_untraceable,
    }
    if "entry_ok" in merged.columns:
        summary["entry_mismatches"] = int((merged["entry_ok"] == False).sum())  # noqa: E712

    return summary, merged


## 2. Chạy trên dữ liệu thật

4 lượt archive đã có sẵn từ phiên kiểm chứng missing-bar 2026-09-01
(`reports/missing-bar-followup-2026-09-01.md`) — dùng làm ví dụ/tự-kiểm-tra
cho chính notebook này (đã biết trước đáp án đúng từ phân tích PowerShell
thủ công, nên nếu notebook ra số khác thì bản thân notebook có bug).

In [3]:
CBOTS_DATA = Path.home() / "Documents" / "cAlgo" / "Data" / "cBots"

RUNS = [
    {
        "strategy": "combo",
        "csv": r"Z:\Desktop\og_program\runtime\exports\combo_US30_H4_full_history_signals.csv",
        "run_dir": CBOTS_DATA / "Combo" / "1bae1864-5246-4220-b6c4-e8fd3dbaae4e-Default" / "ArchivedRuns" / "US30_H4_FallbackOFF_2026short_CANONICAL_20260901-0025",
    },
    {
        "strategy": "combo",
        "csv": r"Z:\Desktop\og_program\runtime\exports\combo_US30_H4_full_history_signals.csv",
        "run_dir": CBOTS_DATA / "Combo" / "1bae1864-5246-4220-b6c4-e8fd3dbaae4e-Default" / "ArchivedRuns" / "US30_H4_FallbackON_2026short_20260901-0035",
    },
    {
        "strategy": "ma_cross",
        "csv": r"Z:\Desktop\og_program\runtime\exports\ma_cross_US30_M30_full_history_signals.csv",
        "run_dir": CBOTS_DATA / "MA Cross" / "a08e1adc-bff4-4dc4-8156-9993c09b0ecb-Default" / "ArchivedRuns" / "US30_M30_FallbackOFF_2026short_20260901-0013",
    },
    {
        "strategy": "ma_cross",
        "csv": r"Z:\Desktop\og_program\runtime\exports\ma_cross_US30_M30_full_history_signals.csv",
        "run_dir": CBOTS_DATA / "MA Cross" / "a08e1adc-bff4-4dc4-8156-9993c09b0ecb-Default" / "ArchivedRuns" / "US30_M30_FallbackON_2026short_20260901-0015",
    },
    {
        "strategy": "combo",
        "csv": r"Z:\Desktop\og_program\runtime\exports\combo_US30_H4_full_history_signals.csv",
        "run_dir": CBOTS_DATA / "Combo" / "1bae1864-5246-4220-b6c4-e8fd3dbaae4e-Default" / "ArchivedRuns" / "US30_H4_AlwaysFallback_2025plus_20260901-0204",
    },
    {
        "strategy": "ma_cross",
        "csv": r"Z:\Desktop\og_program\runtime\exports\ma_cross_US30_M30_full_history_signals.csv",
        "run_dir": CBOTS_DATA / "MA Cross" / "a08e1adc-bff4-4dc4-8156-9993c09b0ecb-Default" / "ArchivedRuns" / "US30_M30_AlwaysFallback_2025plus_20260901-0208",
    },
]

reports = []
merged_frames = {}
for run in RUNS:
    summary, merged = build_fidelity_report(run["csv"], str(run["run_dir"]), run["strategy"])
    reports.append(summary)
    merged_frames[run["run_dir"].name] = merged

report_df = pd.DataFrame(reports).fillna(0)
report_df


,strategy,signal_csv,archived_run,run_window,total_signals_in_csv,before_test_window,placed,in_window_no_exact_bar_fallback_off,direction_mismatches,untraceable_volume_floor_skips,entry_mismatches,rejected
0,combo,combo_US30_H4_full_history_signals.csv,US30_H4_FallbackOFF_2026short_CANONICAL_202609...,2026-01-04 23:05:00.008000 -> 2026-08-26 17:00...,753,691,53,9.0,0,0,0.0,0.0
1,combo,combo_US30_H4_full_history_signals.csv,US30_H4_FallbackON_2026short_20260901-0035,2026-01-04 23:05:00.008000 -> 2026-08-26 17:00...,753,691,62,0.0,0,0,0.0,0.0
2,ma_cross,ma_cross_US30_M30_full_history_signals.csv,US30_M30_FallbackOFF_2026short_20260901-0013,2026-01-02 21:30:00.618000 -> 2026-08-30 23:59...,2934,2731,123,3.0,0,0,0.0,77.0
3,ma_cross,ma_cross_US30_M30_full_history_signals.csv,US30_M30_FallbackON_2026short_20260901-0015,2026-01-02 21:30:00.618000 -> 2026-08-30 23:59...,2934,2731,124,0.0,0,0,0.0,79.0
4,combo,combo_US30_H4_full_history_signals.csv,US30_H4_AlwaysFallback_2025plus_20260901-0204,2025-01-02 14:00:00.233000 -> 2026-08-26 17:00...,753,613,140,0.0,0,0,0.0,0.0
5,ma_cross,ma_cross_US30_M30_full_history_signals.csv,US30_M30_AlwaysFallback_2025plus_20260901-0208,2025-01-02 05:00:00.866000 -> 2026-08-30 23:59...,2934,2375,343,0.0,0,0,0.0,216.0


### Ý nghĩa các cột

- `placed` — tín hiệu được cBot nhận đúng và đặt lệnh thành công (mục tiêu chính).
- `rejected` — cBot nhận đúng tín hiệu nhưng bị broker từ chối (thường do margin) — KHÔNG phải lỗi nhận diện signal.
- `fallback_expired_waiting` — cơ chế fallback nhận ra thiếu bar nhưng hết hạn trước khi có tick khớp được.
- `before_test_window` — tín hiệu nằm ngoài khung ngày backtest đang chạy — bình thường, không phải lỗi.
- `in_window_no_exact_bar_fallback_off` — trong khung test, không có bar FTMO khớp đúng, nhưng `Enable Missing-Bar Fallback` đang TẮT nên cBot không hề cố xử lý — ĐÚNG THIẾT KẾ, không phải lỗi (đây là lý do 2 lượt FallbackOFF trong mẫu có số này > 0 mà vẫn hoàn toàn bình thường).
- `⚠ IN_WINDOW_BUT_MISSING` — **tín hiệu nằm TRONG khung test, fallback ĐANG BẬT, nhưng log không hề nhắc tới** — đây mới là cờ báo lỗi thật cần điều tra (0 ở cả 4 lượt mẫu, đúng như kỳ vọng cho hệ thống đang lành mạnh).
- `direction_mismatches` / `entry_mismatches` — số lệnh đặt sai hướng hoặc sai giá entry so với CSV gốc — phải luôn = 0.

## 3. Soi chi tiết 1 lượt — mẫu các tín hiệu KHÔNG phải 'placed'

In [4]:
example_run = RUNS[1]["run_dir"].name  # Combo H4 FallbackON
merged = merged_frames[example_run]
merged[merged["status"] != "placed"].head(20)


,bartime,atr,entry,signal,alignment,outcome,log_direction,log_entry,error,status,direction_ok,entry_ok
0,2017-05-04 13:00:00,61.882273,20829.0,-1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
1,2017-05-05 17:00:00,61.140828,21044.0,1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
2,2017-05-09 17:00:00,44.289361,20925.0,-1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
3,2017-05-15 10:00:00,52.481652,20980.0,1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
4,2017-06-07 13:00:00,47.937613,21108.0,-1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
5,2017-06-08 13:00:00,61.011246,21287.0,1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
6,2017-06-21 13:00:00,53.245132,21404.0,-1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
7,2017-06-26 06:00:00,66.446145,21485.0,1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
8,2017-06-29 13:00:00,87.394077,21267.0,-1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
9,2017-07-03 06:00:00,73.098633,21449.0,1,NaN,NaN,NaN,NaN,NaN,before_test_window,None,None
